# Лекция 12. SQL, SQLite и ORM в Python

## 1. Базы данных и SQLiteStudio

**Вся глава выполняется в SQLiteStudio.** SQL-блоки ниже — текст для копирования в редактор SQL, а не исполняемые ячейки Python.

База данных — организованное хранилище информации. SQLite — встраиваемая СУБД; SQL (Structured Query Language) — язык запросов; SQLiteStudio — графический клиент для работы с SQLite.

В реляционной модели таблица состоит из строк (записей) и столбцов (атрибутов). Первичный ключ (PRIMARY KEY) однозначно идентифицирует запись; внешний ключ (FOREIGN KEY) связывает таблицы.

### 1.1. Подготовка к работе
1. Добавьте существующую базу `northwind_small.sqlite` через меню Database → Add a database.
2. Откройте таблицу Product. Во вкладке Structure посмотрите имена и типы столбцов, во вкладке Data — записи.
3. Откройте редактор SQL (SQL Editor), выберите Northwind в списке подключений.
4. Введите запрос и нажмите кнопку выполнения. Если в редакторе несколько запросов, выделяйте нужный запрос перед запуском.
5. Изучите таблицу результата. SELECT не изменяет исходные данные.

| Таблица | Назначение |
|---|---|
| Product | Товары |
| Category | Категории |
| Customer | Покупатели |
| Supplier | Поставщики |
| Order | Заказы (имя следует заключать в двойные кавычки: `"Order"`) |
| OrderDetail | Позиции заказов |

Далее используем в основном Product: Id — идентификатор, ProductName — название, UnitPrice — цена, UnitsInStock — остаток, CategoryId — идентификатор категории.

### 1.2. SELECT и FROM: первый запрос

SQL — декларативный язык: мы указываем, **какие данные нужны**, а не алгоритм их перебора.

```sql
SELECT *
FROM Product;
```

- `SELECT` — какие столбцы или выражения должны попасть в результат.
- `*` — все столбцы.
- `FROM` — из какой таблицы взять данные.
- `Product` — имя таблицы.
- `;` — конец инструкции.

Ключевые слова SQL нечувствительны к регистру, но для читаемости пишем их заглавными буквами. Переносы строк не меняют смысл запроса.

#### Один столбец
```sql
SELECT ProductName FROM Product;
```

#### Несколько столбцов
```sql
SELECT ProductName, UnitPrice, UnitsInStock
FROM Product;
```

Столбцы перечисляются через запятую. Их порядок определяет порядок столбцов результата, а не порядок строк. Без ORDER BY порядок строк не гарантируется.

**Попробуйте:** выведите только Id и ProductName; затем поменяйте их местами.

### 1.3. Выражения и псевдонимы AS

SELECT может содержать не только имена столбцов, но и вычисления.

```sql
SELECT ProductName, UnitPrice * UnitsInStock
FROM Product;
```

Для каждой строки вычисляется стоимость имеющегося запаса. Исходная таблица не меняется.

```sql
SELECT ProductName AS name,
       UnitPrice AS price,
       UnitPrice * UnitsInStock AS stock_value
FROM Product;
```

`AS` задаёт имя столбца результата: слева исходное поле или выражение, справа псевдоним. Если имя содержит пробел, используйте двойные кавычки. Строковые значения пишутся в одинарных кавычках.

**Попробуйте:** выведите название и цену товара с условной скидкой 10% (`UnitPrice * 0.9`).

### 1.4. WHERE: выбор строк по условию

```sql
SELECT ProductName, UnitPrice
FROM Product
WHERE UnitPrice > 20;
```

WHERE проверяет условие для каждой строки. В результат попадают только строки, для которых оно истинно. SELECT выбирает столбцы, WHERE — строки.

| Оператор | Значение |
|---|---|
| `=` | равно (не Python `==`) |
| `!=` или `<>` | не равно |
| `>` / `<` | больше / меньше |
| `>=` / `<=` | больше или равно / меньше или равно |

```sql
SELECT ProductName, UnitsInStock
FROM Product
WHERE UnitsInStock = 0;
```

Для чисел кавычки не нужны; текст заключаем в одинарные кавычки:

```sql
SELECT CompanyName, Country
FROM Customer
WHERE Country = 'Germany';
```

**Попробуйте:** найдите товары по цене не выше 15.

### 1.5. AND, OR, NOT

`AND` требует выполнения обоих условий, `OR` — хотя бы одного, `NOT` отрицает условие.

```sql
SELECT ProductName, UnitPrice, UnitsInStock
FROM Product
WHERE UnitPrice > 20 AND UnitsInStock > 0;
```

```sql
SELECT CompanyName, Country
FROM Customer
WHERE Country = 'Germany' OR Country = 'France';
```

```sql
SELECT CompanyName, Country
FROM Customer
WHERE NOT (Country = 'Germany');
```

Приоритет: NOT, затем AND, затем OR. Скобки позволяют явно задать порядок.

```sql
SELECT ProductName, UnitPrice, CategoryId
FROM Product
WHERE (CategoryId = 1 OR CategoryId = 2)
  AND UnitPrice > 20;
```

**Попробуйте:** выберите товары дешевле 10 или дороже 50, но только имеющиеся на складе.

### 1.6. BETWEEN и IN

`значение BETWEEN нижняя_граница AND верхняя_граница` проверяет попадание в интервал, включая обе границы.

```sql
SELECT ProductName, UnitPrice
FROM Product
WHERE UnitPrice BETWEEN 10 AND 30;
```

Это эквивалентно `UnitPrice >= 10 AND UnitPrice <= 30`.

`значение IN (значение1, значение2, ...)` проверяет принадлежность набору.

```sql
SELECT CompanyName, Country
FROM Customer
WHERE Country IN ('Germany', 'France');
```

Можно использовать NOT BETWEEN и NOT IN. При наличии NULL в наборе NOT IN требует особой осторожности.

**Попробуйте:** выберите товары категорий 1, 3, 5 с ценой от 15 до 40.

### 1.7. LIKE: поиск по шаблону

Синтаксис: `столбец LIKE 'шаблон'`.

- `%` — ноль или более произвольных символов.
- `_` — ровно один произвольный символ.
- Остальные символы — обычный текст.

```sql
SELECT CompanyName FROM Customer
WHERE CompanyName LIKE 'A%';
```

```sql
SELECT ProductName FROM Product
WHERE ProductName LIKE '%ch%';
```

```sql
SELECT ProductName FROM Product
WHERE ProductName LIKE 'Ch_i';
```

Первый запрос ищет начало строки, второй — подстроку, третий — совпадение с одним неизвестным символом. В SQLite LIKE по умолчанию не различает регистр ASCII-букв, но это не распространяется автоматически на весь Unicode.

**Попробуйте:** найдите товары, название которых заканчивается на `s`.

### 1.8. NULL: отсутствующее значение

NULL — неизвестное или отсутствующее значение. Это не ноль и не пустая строка. Проверки `= NULL` и `!= NULL` не подходят: сравнение с NULL обычно даёт неизвестный результат.

```sql
SELECT CompanyName, Region
FROM Customer
WHERE Region IS NULL;
```

```sql
SELECT CompanyName, Region
FROM Customer
WHERE Region IS NOT NULL;
```

Пустая строка проверяется отдельно: `Region = ''`. Некоторые базы используют пустые строки вместо NULL; отсутствие строк в результате тоже допустимо.

### 1.9. ORDER BY: сортировка

```sql
SELECT ProductName, UnitPrice
FROM Product
ORDER BY UnitPrice;
```

После ORDER BY указываются столбец или выражение и необязательное направление:
- `ASC` — по возрастанию (по умолчанию).
- `DESC` — по убыванию.

```sql
SELECT ProductName, UnitPrice
FROM Product
ORDER BY UnitPrice DESC;
```

Сортировка по нескольким критериям: сначала цена, при равных ценах — название, затем уникальный Id.

```sql
SELECT ProductName, UnitPrice
FROM Product
ORDER BY UnitPrice DESC, ProductName ASC, Id ASC;
```

**Попробуйте:** отсортируйте имеющиеся на складе товары по остатку от меньшего к большему.

### 1.10. LIMIT и OFFSET

`LIMIT n` — не более n строк результата; `OFFSET k` — пропустить первые k строк.

```sql
SELECT ProductName, UnitPrice
FROM Product
ORDER BY UnitPrice DESC, Id
LIMIT 5;
```

Следующие пять товаров:

```sql
SELECT ProductName, UnitPrice
FROM Product
ORDER BY UnitPrice DESC, Id
LIMIT 5 OFFSET 5;
```

LIMIT применяется к результату после фильтрации и сортировки. Для воспроизводимой выборки задавайте ORDER BY с однозначным порядком.

**Попробуйте:** найдите третий по цене товар (от дорогих к дешёвым).

### 1.11. DISTINCT

```sql
SELECT Country FROM Customer;
```

В результате страны могут повторяться. DISTINCT убирает дубликаты строк результата.

```sql
SELECT DISTINCT Country FROM Customer;
```

```sql
SELECT DISTINCT Country, City
FROM Customer
ORDER BY Country, City;
```

Во втором случае уникальны именно пары (страна, город), а не каждый столбец отдельно.

**Промежуточная практика:** получите список уникальных стран покупателей, начинающихся на A, в алфавитном порядке.

### 1.12. Агрегатные функции: COUNT

Обычный SELECT возвращает отдельные записи. Агрегатная функция обрабатывает набор строк и возвращает одно значение для всего набора или группы.

```sql
SELECT COUNT(*) FROM Product;
```

Параметры COUNT:
- `COUNT(*)` — число строк, включая строки с NULL.
- `COUNT(столбец)` — число значений, отличных от NULL.
- `COUNT(DISTINCT столбец)` — число различных значений, отличных от NULL.

```sql
SELECT COUNT(*) AS customers,
       COUNT(Region) AS known_regions,
       COUNT(DISTINCT Country) AS countries
FROM Customer;
```

```sql
SELECT COUNT(*) AS expensive_products
FROM Product
WHERE UnitPrice > 20;
```

Сначала отбираются строки по WHERE, затем считается их количество.

### 1.13. SUM, AVG, MIN, MAX и ROUND

| Функция | Аргумент | Результат |
|---|---|---|
| SUM(x) | числовой столбец или выражение | сумма |
| AVG(x) | числовой столбец или выражение | среднее |
| MIN(x) | столбец или выражение | минимум |
| MAX(x) | столбец или выражение | максимум |
| ROUND(x, n) | число x, число знаков n | округлённое число |

```sql
SELECT SUM(UnitsInStock) AS total_units FROM Product;
```

```sql
SELECT AVG(UnitPrice) AS average_price FROM Product;
```

```sql
SELECT MIN(UnitPrice) AS min_price,
       MAX(UnitPrice) AS max_price
FROM Product;
```

```sql
SELECT ROUND(AVG(UnitPrice), 2) AS average_price,
       SUM(UnitPrice * UnitsInStock) AS stock_value
FROM Product;
```

ROUND — обычная скалярная функция, не агрегатная. Агрегаты выше игнорируют NULL; при отсутствии ненулевых (не NULL) значений SUM, AVG, MIN, MAX возвращают NULL, COUNT — 0.

**Попробуйте:** вычислите среднюю цену товаров, которые есть на складе.

### 1.14. GROUP BY: группировка

Предположим, нужно посчитать покупателей не вообще, а отдельно по каждой стране.

```sql
SELECT Country, COUNT(*) AS amount
FROM Customer
GROUP BY Country;
```

GROUP BY принимает один или несколько столбцов (или выражений). Строки с одинаковыми значениями ключей образуют группу. Агрегат рассчитывается отдельно для каждой группы.

```sql
SELECT CategoryId, ROUND(AVG(UnitPrice), 2) AS average_price
FROM Product
GROUP BY CategoryId
ORDER BY average_price DESC;
```

```sql
SELECT Country, City, COUNT(*) AS amount
FROM Customer
GROUP BY Country, City;
```

При группировке перечисляйте в SELECT ключи группировки и агрегатные выражения. SQLite допускает некоторые другие столбцы, но их значения могут быть неоднозначными.

**Попробуйте:** найдите количество товаров и суммарный остаток каждой категории.

### 1.15. HAVING: фильтрация групп

WHERE проверяет отдельные строки до группировки, HAVING — группы после вычисления агрегатов.

```sql
SELECT Country, COUNT(*) AS amount
FROM Customer
GROUP BY Country
HAVING COUNT(*) > 5;
```

Обе конструкции можно объединять:

```sql
SELECT CategoryId, COUNT(*) AS amount
FROM Product
WHERE UnitsInStock > 0
GROUP BY CategoryId
HAVING COUNT(*) >= 5
ORDER BY amount DESC;
```

Запрос оставляет товары в наличии, группирует их по категории, выбирает категории с пятью и более такими товарами и сортирует результат.

**Порядок записи основных частей:** SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT → OFFSET.

Упрощённый логический порядок обработки: FROM → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT/OFFSET.

### 1.16. Подзапросы

Сначала найдём среднюю цену:

```sql
SELECT AVG(UnitPrice) FROM Product;
```

Теперь используем результат как правую часть сравнения:

```sql
SELECT ProductName, UnitPrice
FROM Product
WHERE UnitPrice > (
    SELECT AVG(UnitPrice) FROM Product
);
```

Внутренний запрос в скобках возвращает одно значение. Внешний запрос сравнивает с ним цены. Для подзапроса, возвращающего набор значений, удобно использовать IN:

```sql
SELECT ProductName FROM Product
WHERE CategoryId IN (
    SELECT Id FROM Category WHERE CategoryName LIKE 'C%'
);
```

**Попробуйте:** найдите товары с остатком выше среднего.

### 1.17. JOIN: связь таблиц

В Product хранится CategoryId, а в Category — Id и CategoryName. Вместо дублирования названия категории используем внешний ключ.

```sql
SELECT Id, CategoryName FROM Category;
```

```sql
SELECT ProductName, CategoryId FROM Product;
```

Теперь объединим таблицы по соответствующим ключам:

```sql
SELECT Product.ProductName, Category.CategoryName
FROM Product
INNER JOIN Category
ON Product.CategoryId = Category.Id;
```

- FROM — исходная таблица.
- INNER JOIN (можно просто JOIN) — присоединяемая таблица; оставляет только совпадающие пары строк.
- ON — условие соответствия строк.
- `Таблица.Столбец` — уточнение, из какой таблицы взят столбец.

Псевдонимы таблиц сокращают запись:

```sql
SELECT p.ProductName, c.CategoryName, p.UnitPrice
FROM Product AS p
JOIN Category AS c ON p.CategoryId = c.Id
WHERE p.UnitPrice > 20;
```

**Попробуйте:** аналогично присоедините Supplier по Product.SupplierId = Supplier.Id и выведите название товара и поставщика (CompanyName).

### 1.18. LEFT JOIN

LEFT JOIN сохраняет все строки левой таблицы, даже если справа нет соответствия; столбцы правой таблицы тогда содержат NULL.

```sql
SELECT c.CategoryName, p.ProductName
FROM Category AS c
LEFT JOIN Product AS p ON c.Id = p.CategoryId;
```

```sql
SELECT c.CategoryName, COUNT(p.Id) AS amount
FROM Category AS c
LEFT JOIN Product AS p ON c.Id = p.CategoryId
GROUP BY c.Id, c.CategoryName;
```

COUNT(p.Id) считает только существующие товары. COUNT(*) посчитал бы и строку категории без товара. В Northwind пустых категорий может не быть, поэтому разница между JOIN и LEFT JOIN не обязательно видна на этих данных.

**Попробуйте:** выведите среднюю цену по каждой категории с её названием.

### 1.19. CREATE TABLE: создаём собственную базу

В SQLiteStudio создайте отдельный файл `studio_people.db` и выберите его в SQL Editor. Не изменяйте Northwind.

```sql
PRAGMA foreign_keys = ON;
```

Эта настройка включает проверку внешних ключей для текущего соединения.

```sql
CREATE TABLE students (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    surname TEXT NOT NULL
);
```

CREATE TABLE принимает имя таблицы и список описаний столбцов в скобках. Каждое описание содержит имя, тип и необязательные ограничения. INTEGER — целые, REAL — вещественные, TEXT — текст, BLOB — двоичные данные; NULL обозначает отсутствие значения. В обычных таблицах SQLite типы задают предпочтительный тип хранения, а не всегда строгое ограничение.

```sql
CREATE TABLE grade (
    id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL REFERENCES students(id),
    mark INTEGER NOT NULL CHECK(mark BETWEEN 1 AND 10)
);
```

PRIMARY KEY — первичный ключ; INTEGER PRIMARY KEY позволяет автоматически назначать Id. NOT NULL запрещает NULL. REFERENCES задаёт внешний ключ. CHECK проверяет условие. Одна запись students может иметь множество записей grade.

### 1.20. INSERT: добавление записей

```sql
INSERT INTO students (name, surname)
VALUES ('Иван', 'Иванов');
```

После имени таблицы перечисляем столбцы, после VALUES — соответствующие значения в том же порядке. Id назначается автоматически.

```sql
INSERT INTO students (name, surname)
VALUES ('Анна', 'Петрова'), ('Пётр', 'Сидоров');
```

```sql
SELECT * FROM students;
```

```sql
INSERT INTO grade (student_id, mark)
VALUES (1, 8), (1, 10), (2, 9);
```

Последний пример предполагает новую базу и Id 1 и 2; проверьте их перед выполнением. Повторное выполнение INSERT добавляет новые записи.

### 1.21. UPDATE: изменение записей

```sql
UPDATE students
SET name = 'Александр'
WHERE id = 1;
```

UPDATE задаёт таблицу, SET — столбцы и новые значения, WHERE — строки для изменения. Несколько присваиваний разделяются запятой.

```sql
UPDATE students
SET name = 'Анна', surname = 'Орлова'
WHERE id = 2;
```

Перед UPDATE полезно выполнить SELECT с тем же WHERE и проверить выбранные строки. Без WHERE изменятся все записи!

### 1.22. DELETE: удаление записей

```sql
DELETE FROM students WHERE id = 3;
```

DELETE FROM задаёт таблицу, WHERE — условие удаления. Без WHERE удаляются все строки. При включённых внешних ключах нельзя удалить студента, на которого ссылаются оценки, пока ссылки не устранены.

```sql
SELECT * FROM students;
```

CRUD: Create — INSERT, Read — SELECT, Update — UPDATE, Delete — DELETE.

### 1.23. Практика в SQLiteStudio

Выполните самостоятельно на Northwind:
1. Выведите название товара и цену.
2. Выберите товары дешевле 15.
3. Выведите товары категорий 1 и 2, имеющиеся на складе.
4. Найдите названия товаров, начинающиеся с C.
5. Найдите три самых дорогих товара в наличии.
6. Выведите уникальные страны покупателей по алфавиту.
7. Посчитайте количество товаров в наличии.
8. Вычислите среднюю цену товаров с округлением до двух знаков.
9. Найдите страны, в которых не менее трёх покупателей.
10. Найдите товары дешевле средней цены.
11. Выведите название товара и его категории.
12. Выведите названия категорий и среднюю цену товаров каждой категории.

Все запросы выполняются только в SQLiteStudio. Сохраните их в отдельный SQL-файл через редактор. Для следующей главы не требуется переносить результаты или созданную вручную базу: Python создаст отдельную учебную базу.

## 2. SQL из Python: sqlite3

В первой главе мы писали SQL в SQLiteStudio. Теперь те же запросы будет выполнять программа Python. Модуль `sqlite3` входит в стандартную библиотеку.

Используем существующие файлы `northwind_small.sqlite` и `studio_people.db` (созданный в главе 1). Поместите их рядом с ноутбуком и запускайте примеры из этой папки. Новых баз и таблиц в этой главе не создаём. Важно: если ошибиться в имени файла, `connect()` может создать пустую базу. Перед изменением данных завершите открытые транзакции в SQLiteStudio. Повторное выполнение INSERT добавляет новые записи.

### 2.1. Подключение к базе

`sqlite3.connect(database)` принимает путь к файлу базы и возвращает объект соединения. `connection.cursor()` создаёт курсор — объект, через который выполняются SQL-запросы.

In [1]:
import sqlite3

connection = sqlite3.connect("northwind_small.sqlite")
cursor = connection.cursor()

### 2.2. Первый запрос

`cursor.execute(sql)` принимает строку с SQL-запросом. После SELECT метод `fetchall()` возвращает список кортежей: один кортеж соответствует одной строке результата. Порядок элементов совпадает с порядком столбцов в SELECT.

In [2]:
cursor.execute("SELECT ProductName, UnitPrice FROM Product LIMIT 5")
result = cursor.fetchall()
print(result)

for name, price in result:
    print(name, price)

connection.close()

[('Chai', 18), ('Chang', 19), ('Aniseed Syrup', 10), ("Chef Anton's Cajun Seasoning", 22), ("Chef Anton's Gumbo Mix", 21.35)]
Chai 18
Chang 19
Aniseed Syrup 10
Chef Anton's Cajun Seasoning 22
Chef Anton's Gumbo Mix 21.35


`close()` закрывает соединение. В следующих примерах будем открывать и закрывать соединение заново, чтобы каждый пример можно было запустить отдельно после импорта sqlite3.

### 2.3. Получение результатов

- `fetchone()` — одна следующая строка или `None`, если строк больше нет.
- `fetchmany(n)` — до n следующих строк.
- `fetchall()` — все оставшиеся строки.

Методы последовательно считывают результат одного запроса. Для повторного чтения нужно снова выполнить `execute()`.

In [3]:
connection = sqlite3.connect("northwind_small.sqlite")
cursor = connection.cursor()
cursor.execute("SELECT ProductName FROM Product ORDER BY Id LIMIT 5")
print(cursor.fetchone())
print(cursor.fetchmany(2))
print(cursor.fetchall())
connection.close()

('Chai',)
[('Chang',), ('Aniseed Syrup',)]
[("Chef Anton's Cajun Seasoning",), ("Chef Anton's Gumbo Mix",)]


### 2.4. Параметры SQL-запроса

`execute(sql, parameters)` принимает SQL и значения параметров отдельно. Знак `?` обозначает место значения; параметры передаются кортежем в порядке появления знаков `?`. Для одного элемента обязательна запятая: `(country,)`. Не используйте f-строки для подстановки пользовательских значений: это может привести к SQL-инъекции.

In [4]:
country = "Germany"
connection = sqlite3.connect("northwind_small.sqlite")
cursor = connection.cursor()
cursor.execute("SELECT CompanyName FROM Customer WHERE Country = ?", (country,))
for row in cursor.fetchall():
    print(row[0])
connection.close()

Alfreds Futterkiste
Blauer See Delikatessen
Drachenblut Delikatessen
Frankenversand
Königlich Essen
Lehmanns Marktstand
Morgenstern Gesundkost
Ottilies Käseladen
QUICK-Stop
Toms Spezialitäten
Die Wandernde Kuh


In [ ]:
min_price = 10
max_price = 20
connection = sqlite3.connect("northwind_small.sqlite")
cursor = connection.cursor()
cursor.execute("SELECT ProductName, UnitPrice FROM Product WHERE UnitPrice BETWEEN ? AND ?", (min_price, max_price))
print(cursor.fetchall())
connection.close()

### 2.5. Изменение данных: INSERT и commit

Переходим к `studio_people.db`. После INSERT, UPDATE или DELETE вызываем `connection.commit()`, чтобы зафиксировать изменения. `cursor.lastrowid` возвращает идентификатор последней вставленной записи в нашем примере.

In [ ]:
connection = sqlite3.connect("studio_people.db")
cursor = connection.cursor()
cursor.execute("INSERT INTO students (name, surname) VALUES (?, ?)", ("Мария", "Соколова"))
student_id = cursor.lastrowid
connection.commit()
print(student_id)
connection.close()

### 2.6. UPDATE и DELETE

В WHERE используем идентификатор созданного выше студента, чтобы не изменить чужие записи. Запустите предыдущий пример прежде, чем выполнять эти два.

In [ ]:
connection = sqlite3.connect("studio_people.db")
cursor = connection.cursor()
cursor.execute("UPDATE students SET name = ? WHERE id = ?", ("Марина", student_id))
connection.commit()
connection.close()

In [ ]:
connection = sqlite3.connect("studio_people.db")
cursor = connection.cursor()
cursor.execute("PRAGMA foreign_keys = ON")
cursor.execute("DELETE FROM students WHERE id = ?", (student_id,))
connection.commit()
connection.close()

### 2.7. Добавление нескольких записей

`executemany(sql, parameters)` выполняет один запрос для каждого набора параметров из последовательности.

In [ ]:
connection = sqlite3.connect("studio_people.db")
cursor = connection.cursor()
students = [("Олег", "Орлов"), ("Елена", "Миронова")]
cursor.executemany("INSERT INTO students (name, surname) VALUES (?, ?)", students)
connection.commit()
connection.close()

### 2.8. Транзакции и обработка ошибок

Добавление студента и его первой оценки должно выполняться как единое целое. `commit()` сохраняет обе операции, `rollback()` отменяет незавершённые изменения. `PRAGMA foreign_keys = ON` включает проверку внешних ключей для данного соединения. `finally` гарантирует закрытие соединения даже при ошибке.

In [ ]:
connection = sqlite3.connect("studio_people.db")
cursor = connection.cursor()
cursor.execute("PRAGMA foreign_keys = ON")
try:
    cursor.execute("INSERT INTO students (name, surname) VALUES (?, ?)", ("Анна", "Волкова"))
    cursor.execute("INSERT INTO grade (student_id, mark) VALUES (?, ?)", (cursor.lastrowid, 8))
    connection.commit()
except sqlite3.Error as error:
    connection.rollback()
    print(error)
finally:
    connection.close()

### 2.9. Функции для работы с базой

Когда механизм понятен, можно перенести повторяющиеся операции в функции. Каждая функция выполняет конкретную задачу, не скрывая SQL за универсальной обёрткой.

In [ ]:
def get_students():
    connection = sqlite3.connect("studio_people.db")
    try:
        cursor = connection.cursor()
        cursor.execute("SELECT id, name, surname FROM students ORDER BY id")
        return cursor.fetchall()
    finally:
        connection.close()


def add_student(name, surname):
    connection = sqlite3.connect("studio_people.db")
    try:
        cursor = connection.cursor()
        cursor.execute("INSERT INTO students (name, surname) VALUES (?, ?)", (name, surname))
        connection.commit()
        return cursor.lastrowid
    finally:
        connection.close()

print(get_students())

### Практика 2. sqlite3

1. Напишите функцию `get_products(max_price)`, возвращающую товары дешевле указанной цены.
2. Напишите функцию `add_grade(student_id, mark)` с параметризованным запросом и фиксацией изменений.
3. Напишите функцию `get_grades(student_id)`, возвращающую оценки студента.

In [ ]:
# Задание 1
# Ваш код

In [ ]:
# Задание 2
# Ваш код

In [ ]:
# Задание 3
# Ваш код

## 3. ORM и SQLAlchemy

ORM (Object-Relational Mapping) сопоставляет классы с таблицами, экземпляры — со строками, атрибуты — со столбцами. SQL не исчезает: библиотека генерирует запросы.

Установка в текущем окружении Jupyter: раскомментируйте следующую команду, если библиотека не установлена. Используем API SQLAlchemy 2.x (`select`, `Session.get`).

In [ ]:
# %pip install "sqlalchemy>=2,<3"

### 3.1 Описание моделей
`declarative_base()` создаёт базовый класс. `Column` описывает столбец, `ForeignKey` — внешний ключ, `relationship` — связь объектов. `back_populates` синхронизирует обе стороны отношения. В отличие от исходной модели, оценка имеет тип Integer, а не String.

In [5]:
import sqlalchemy as sa
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

Base = declarative_base()

class Student(Base):
    __tablename__ = 'students'
    id = sa.Column(sa.Integer, primary_key=True)
    name = sa.Column(sa.String, nullable=False)
    surname = sa.Column(sa.String, nullable=False)
    grades = relationship('Grade', back_populates='student')

class Grade(Base):
    __tablename__ = 'grade'
    __table_args__ = (sa.CheckConstraint('mark BETWEEN 1 AND 10'),)
    id = sa.Column(sa.Integer, primary_key=True)
    student_id = sa.Column(sa.Integer, sa.ForeignKey('students.id'), nullable=False)
    mark = sa.Column(sa.Integer, nullable=False)
    student = relationship('Student', back_populates='grades')

### 3.2 Engine и Session
Engine управляет соединениями; Session отслеживает объекты и транзакции; sessionmaker создаёт сессии. `create_all()` создаёт отсутствующие таблицы, но не мигрирует существующие. Подключаем ту же учебную базу.

Используем те же таблицы из главы 1. В этом примере `create_all()` не вызываем: структура уже создана в SQLiteStudio.

In [6]:
engine = sa.create_engine("sqlite:///studio_people.db", echo=False)

@sa.event.listens_for(engine, 'connect')
def enable_foreign_keys(dbapi_connection, connection_record):
    dbapi_connection.execute('PRAGMA foreign_keys = ON')

Session = sessionmaker(bind=engine)

### 3.3 CREATE
Создаём объект, добавляем его в сессию, сохраняем. Для демонстрации включим echo: увидим SQL, который ORM формирует самостоятельно.

In [7]:
engine.echo = True
with Session() as session:
    student = Student(name='Дмитрий', surname='Козлов')
    session.add(student)
    session.commit()
    new_id = student.id
engine.echo = False

2026-11-25 08:35:50,309 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-11-25 08:35:50,310 INFO sqlalchemy.engine.Engine INSERT INTO students (name, surname) VALUES (?, ?)
2026-11-25 08:35:50,311 INFO sqlalchemy.engine.Engine [generated in 0.00053s] ('Дмитрий', 'Козлов')
2026-11-25 08:35:50,312 INFO sqlalchemy.engine.Engine ROLLBACK


OperationalError: (sqlite3.OperationalError) no such table: students
[SQL: INSERT INTO students (name, surname) VALUES (?, ?)]
[parameters: ('Дмитрий', 'Козлов')]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

### 3.4 READ
`select(Student)` задаёт запрос, `scalars()` возвращает объекты вместо строк результата.

In [ ]:
with Session() as session:
    students = session.scalars(sa.select(Student).order_by(Student.id)).all()
    for student in students:
        print(student.id, student.name, student.surname)

In [ ]:
with Session() as session:
    query = sa.select(Student).where(Student.surname == 'Иванов').order_by(Student.id).limit(5)
    for student in session.scalars(query):
        print(student.name, student.surname)
    print(session.get(Student, new_id).name)

### 3.5 UPDATE
Меняем атрибут объекта и фиксируем транзакцию.

In [ ]:
with Session.begin() as session:
    student = session.get(Student, new_id)
    student.name = 'Денис'
with Session() as session:
    for student in session.scalars(sa.select(Student).order_by(Student.id)):
        print(student.id, student.name, student.surname)

### 3.6 DELETE
Удаляем созданного выше студента, у которого ещё нет оценок.

In [ ]:
with Session.begin() as session:
    student = session.get(Student, new_id)
    if student is not None:
        session.delete(student)
with Session() as session:
    for student in session.scalars(sa.select(Student).order_by(Student.id)):
        print(student.id, student.name, student.surname)

### 3.7 Связи объектов
Оценки можно добавлять в коллекцию `student.grades`. ORM установит внешний ключ при сохранении.

In [ ]:
with Session.begin() as session:
    student = Student(name='Анна', surname='Лебедева')
    student.grades.extend([Grade(mark=8), Grade(mark=10)])
    session.add(student)
    session.flush()  # выполняет INSERT, но ещё не фиксирует транзакцию
    student_id = student.id

In [ ]:
with Session() as session:
    student = session.get(Student, student_id)
    marks = [grade.mark for grade in student.grades]
    print(student.name, student.surname, marks)
    if marks:
        print('Средний балл:', sum(marks) / len(marks))
    print('Владелец первой оценки:', student.grades[0].student.name)

### 3.8 JOIN и агрегаты в ORM
Средняя оценка каждого студента, у которого есть оценки. Выражения ORM выполняются в базе, а не в цикле Python.

In [ ]:
with Session() as session:
    query = (sa.select(Student.name, Student.surname, sa.func.avg(Grade.mark))
             .join(Grade, Grade.student_id == Student.id)
             .group_by(Student.id, Student.name, Student.surname))
    for name, surname, average in session.execute(query):
        print(name, surname, average)

### 3.9 Разделение проекта на модули

После знакомства с одним файлом перейдите к структуре приложенного проекта:

```text
project/
    main.py
    db/people.db
    data/
        __init__.py
        db_sessions.py
        __all_models.py
        student.py
        grade.py
```

`db_sessions.py` содержит Base, Engine и фабрику сессий; `student.py` и `grade.py` — модели; `__all_models.py` импортирует обе модели до `metadata.create_all`; `main.py` — бизнес-логика. Для моделей используйте согласованные имена `grades` и `student` в обоих `back_populates`. Не забывайте создать папку db перед подключением.

В исходном проекте применяется `session.query(Student).filter(...)`; это прежний стиль SQLAlchemy, которому в новом коде соответствует `session.scalars(sa.select(Student).where(...))`.

### Практика 3. ORM

1. Добавьте студента и три оценки в одной транзакции.
2. Найдите всех студентов по заданной фамилии и выведите оценки каждого.
3. Измените оценку по её идентификатору.
4. Выведите студентов со средним баллом не ниже 8 (JOIN, GROUP BY, HAVING).

## 3.10. Справочник по описанию моделей SQLAlchemy

```python
from sqlalchemy import Column, Integer, String
from sqlalchemy.orm import relationship
from .db_sessions import Base

class Student(Base):
    __tablename__ = 'students'

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    surname = Column(String, nullable=False)

    grades = relationship('Grade', back_populates='student')
```

- `Base` — базовый класс моделей, созданный через `declarative_base()`.
- `__tablename__` — имя таблицы в базе данных.
- `Column()` — описание столбца: тип данных и дополнительные параметры.
- `relationship()` — связь между объектами моделей, а не новый столбец таблицы.

Класс соответствует таблице, объект — записи, атрибут `Column` — столбцу. Примеры ниже предназначены для чтения и переноса в файлы моделей в среде разработки, а не для повторного объявления моделей в ноутбуке.

### 3.10.1. Типы данных

Типы необходимо импортировать из `sqlalchemy`.

| Тип SQLAlchemy | Тип Python | Пример |
|---|---|---|
| `Integer` | `int` | `age = Column(Integer)` |
| `BigInteger` | `int` | `vk_id = Column(BigInteger)` |
| `Float` | `float` | `rating = Column(Float)` |
| `Numeric(10, 2)` | `Decimal` | `price = Column(Numeric(10, 2))` |
| `String` | `str` | `name = Column(String)` |
| `String(50)` | `str` | `name = Column(String(50))` |
| `Text` | `str` | `description = Column(Text)` |
| `Boolean` | `bool` | `is_active = Column(Boolean)` |
| `Date` | `date` | `birthday = Column(Date)` |
| `Time` | `time` | `start = Column(Time)` |
| `DateTime` | `datetime` | `created_at = Column(DateTime)` |
| `LargeBinary` | `bytes` | `photo = Column(LargeBinary)` |
| `JSON` | JSON-совместимые значения | `settings = Column(JSON)` |

`String(50)` объявляет длину строки, а `Numeric(10, 2)` — точность (10 цифр, из них 2 после точки). SQLite не обеспечивает эти ограничения автоматически.

### 3.10.2. Основные параметры Column()

| Параметр | Назначение | Пример |
|---|---|---|
| `primary_key` | Первичный ключ | `id = Column(Integer, primary_key=True)` |
| `nullable` | Разрешено ли значение NULL | `name = Column(String, nullable=False)` |
| `unique` | Запрет повторяющихся значений | `email = Column(String, unique=True)` |
| `default` | Значение по умолчанию при вставке через SQLAlchemy | `balance = Column(Integer, default=0)` |
| `server_default` | Значение по умолчанию на уровне БД | `balance = Column(Integer, server_default='0')` |
| `autoincrement` | Автоматическая генерация целочисленного ключа | `id = Column(Integer, primary_key=True, autoincrement=True)` |
| `index` | Создание индекса | `surname = Column(String, index=True)` |
| `name` | Имя столбца в БД | `name = Column(String, name='first_name')` |
| `onupdate` | Значение при обновлении через SQLAlchemy | `updated_at = Column(DateTime, onupdate=datetime.now)` |

- Для обычного целочисленного первичного ключа SQLite достаточно `primary_key=True`.
- `nullable=False` запрещает NULL, но не пустую строку.
- `unique=True` не заменяет первичный ключ; SQLite допускает несколько NULL в уникальном столбце.
- Индекс ускоряет некоторые запросы, но занимает место и увеличивает стоимость записи.
- Для текущего времени передают функцию `datetime.now`, а не результат её вызова `datetime.now()`.

```python
from datetime import datetime
from sqlalchemy import Column, DateTime, Boolean

created_at = Column(DateTime, default=datetime.now)
updated_at = Column(DateTime, default=datetime.now, onupdate=datetime.now)
is_active = Column(Boolean, default=True)
```

### 3.10.3. Связи между таблицами

В модели `Grade`:

```python
student_id = Column(Integer, ForeignKey('students.id'), nullable=False)
student = relationship('Student', back_populates='grades')
```

В модели `Student`:

```python
grades = relationship('Grade', back_populates='student')
```

`ForeignKey('students.id')` принимает имя **таблицы и столбца**. `relationship('Student')` принимает имя **класса**. `back_populates` указывает имя атрибута обратной связи в другой модели.

| Параметр relationship() | Назначение |
|---|---|
| `argument` | Связанный класс |
| `back_populates` | Атрибут обратной связи |
| `uselist` | Коллекция или один объект |
| `cascade` | Распространение операций на связанные объекты |
| `order_by` | Сортировка связанных объектов |
| `secondary` | Промежуточная таблица для связи многие ко многим |
| `lazy` | Стратегия загрузки |
| `foreign_keys` | Выбор внешнего ключа при неоднозначности |

Для связи один ко многим достаточно `ForeignKey`, `relationship` и `back_populates`.

```python
student = session.get(Student, 1)
if student is not None:
    student.grades.append(Grade(mark=8))
    session.commit()

    for grade in student.grades:
        print(grade.mark)
```

Каскадное удаление можно настроить через `cascade='all, delete-orphan'`: при удалении студента ORM удалит связанные оценки. Это не то же самое, что SQL `ON DELETE CASCADE`.

### 3.10.4. Дополнительные ограничения

```python
from sqlalchemy import CheckConstraint, UniqueConstraint

mark = Column(Integer, CheckConstraint('mark BETWEEN 1 AND 10'), nullable=False)
```

`CheckConstraint` проверяет условие на уровне базы данных. Для ограничения, относящегося к нескольким столбцам, внутри класса используют `__table_args__`:

```python
__table_args__ = (
    UniqueConstraint('name', 'surname', 'birthday'),
)
```

Это ограничивает повторение комбинации значений (при отсутствии NULL).

**Важно:** изменение модели Python не изменяет автоматически уже существующую таблицу. `Base.metadata.create_all(engine)` создаёт отсутствующие таблицы, но не выполняет миграции.

### 3.10.5. Пример модели пользователя VK-бота

```python
from datetime import datetime
from sqlalchemy import Column, Integer, String, Boolean, DateTime
from .db_sessions import Base

class User(Base):
    __tablename__ = 'users'

    id = Column(Integer, primary_key=True)
    vk_id = Column(Integer, unique=True, nullable=False)
    name = Column(String)
    state = Column(String, default='menu')
    balance = Column(Integer, default=0)
    is_admin = Column(Boolean, default=False)
    created_at = Column(DateTime, default=datetime.now)
```

`vk_id` связывает запись с пользователем VK, `state` хранит состояние диалога, `created_at` — время регистрации. Состояние диалога можно сохранять в БД вместо словаря Python, чтобы оно не терялось после перезапуска бота.

Для первого проекта достаточно `Column`, основных типов, `primary_key`, `nullable`, `unique`, `default`, `ForeignKey` и `relationship`. Остальные параметры добавляются только при необходимости.